# Notebook 02: Predictor set, development folds, pilot sample and sealed 2025 labels

**Study:** predicting recorded road casualty severity (KSI) in Great Britain. This notebook implements sections 3.6.3 and 3.7.1 of the methodology.

**What it does**
1. Reads the eligible casualties saved by notebook 01.
2. Defines the predictor set from the *current* fields only, with a rule-based availability check by year.
3. Derives hour, day of week and month from the date and time fields.
4. Recodes the "-1" missing code to a true missing value for numeric fields only. Categorical fields keep -1 as its own level.
5. Saves **five collision-grouped, outcome-stratified folds** for 2023-2024 and a **fixed 20% pilot sample of collisions**.
6. Writes the 2025 features **without labels**, and seals the 2025 labels in a separate folder with a checksum.
7. Saves evaluation-only files (road-user group, injury-based flag, force system group, police force) for subgroup and sensitivity work.

**What it does not do:** no model fitting and no learned preprocessing. Imputation medians, rare-category rules, scaling and the age spline are learned inside each training fold in the modelling notebook (Roberts et al., 2017; Kapoor and Narayanan, 2023).

**Do not open the sealed 2025 labels file until the freeze step.**

In [1]:
# 1. Setup -----------------------------------------------------------------
import os, sys, json, hashlib, sqlite3, subprocess, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
try:
    display
except NameError:
    display = print

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/stats19_severity')
else:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './stats19_severity'))

DB_PATH = PROJECT / 'db' / 'stats19.sqlite'
LINK = PROJECT / 'outputs' / 'linkage'
OUT = PROJECT / 'outputs' / 'features'
DATA = PROJECT / 'data' / 'model_inputs'
SEALED = PROJECT / 'data' / 'sealed_2025_labels'
ENV = PROJECT / 'environment'
for p in (OUT, DATA, SEALED, ENV):
    p.mkdir(parents=True, exist_ok=True)
assert DB_PATH.exists(), 'Run notebook 01 first: the SQLite database is missing.'

SEED = 20261107
N_FOLDS = 5
PILOT_FRACTION = 0.20
DEV_YEARS, TEST_YEAR = [2023, 2024], 2025
(ENV / 'pip_freeze_notebook02.txt').write_text(
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

Mounted at /content/drive
Python 3.13.18 | pandas 2.2.3 | scikit-learn 1.6.1


In [2]:
# 2. Field dictionary --------------------------------------------------------
# kind: num = numeric (the -1 code becomes missing); cat = categorical code (-1 stays a level).
# group: the "original field" used for permutation importance in 3.7.5.
# CONFIRM the code meanings against the 2025 data guide before Chapter 4.
PRIMARY = [
    # casualty
    ('age_of_casualty', 'casualty', 'num'), ('casualty_imd_decile', 'casualty', 'num'),
    ('casualty_class', 'casualty', 'cat'), ('sex_of_casualty', 'casualty', 'cat'),
    ('casualty_type', 'casualty', 'cat'), ('pedestrian_location', 'casualty', 'cat'),
    ('pedestrian_movement', 'casualty', 'cat'), ('car_passenger', 'casualty', 'cat'),
    ('bus_or_coach_passenger', 'casualty', 'cat'), ('pedestrian_road_maintenance_worker', 'casualty', 'cat'),
    # vehicle (the vehicle the casualty was in; for pedestrians, the linked vehicle reference)
    ('age_of_driver', 'vehicle', 'num'), ('engine_capacity_cc', 'vehicle', 'num'), ('driver_imd_decile', 'vehicle', 'num'),
    ('vehicle_type', 'vehicle', 'cat'), ('towing_and_articulation', 'vehicle', 'cat'),
    ('vehicle_manoeuvre', 'vehicle', 'cat'), ('vehicle_direction_from', 'vehicle', 'cat'),
    ('vehicle_direction_to', 'vehicle', 'cat'), ('vehicle_location_restricted_lane', 'vehicle', 'cat'),
    ('junction_location', 'vehicle', 'cat'), ('skidding_and_overturning', 'vehicle', 'cat'),
    ('hit_object_in_carriageway', 'vehicle', 'cat'), ('vehicle_leaving_carriageway', 'vehicle', 'cat'),
    ('hit_object_off_carriageway', 'vehicle', 'cat'), ('first_point_of_impact', 'vehicle', 'cat'),
    ('vehicle_left_hand_drive', 'vehicle', 'cat'), ('journey_purpose_of_driver', 'vehicle', 'cat'),
    ('sex_of_driver', 'vehicle', 'cat'), ('propulsion_code', 'vehicle', 'cat'),
    ('age_of_vehicle', 'vehicle', 'num'),                    # candidate; the availability rule drops it (empty in 2025)
    # collision
    ('speed_limit', 'collision', 'num'), ('number_of_vehicles', 'collision', 'num'),
    ('number_of_casualties', 'collision', 'num'),          # included; repeated without it (3.6.3)
    ('first_road_class', 'collision', 'cat'), ('road_type', 'collision', 'cat'),
    ('junction_detail', 'collision', 'cat'), ('junction_control', 'collision', 'cat'),
    ('second_road_class', 'collision', 'cat'), ('pedestrian_crossing', 'collision', 'cat'),
    ('light_conditions', 'collision', 'cat'), ('weather_conditions', 'collision', 'cat'),
    ('road_surface_conditions', 'collision', 'cat'), ('carriageway_hazards', 'collision', 'cat'),
    ('urban_or_rural_area', 'collision', 'cat'), ('trunk_road_flag', 'collision', 'cat'),
    # derived from date and time
    ('hour', 'derived', 'cat'), ('day_of_week', 'derived', 'cat'), ('month', 'derived', 'cat'),
]
SENSITIVITY_FIELDS = {'number_of_casualties': 'repeat comparison with this field removed (3.6.3)'}
# Fields reserved for evaluation or sensitivity only, never primary predictors.
RESERVED = {
    'police_force': 'sensitivity run with force included; also used to define unchanged forces (3.7.5)',
    'casualty_injury_based': 'subgroup and sensitivity analysis only (3.6.3)',
    'escooter_flag': 'redundant with casualty_type and vehicle_type',
}
# Excluded by design, with the reason that goes in the dictionary.
EXCLUDED_DESIGN = {
    'collision_severity': 'derived from worst casualty outcome (3.6.3)',
    'casualty_severity': 'outcome',
    'enhanced_casualty_severity': 'severity-derived', 'enhanced_severity_collision': 'severity-derived',
    'casualty_adjusted_severity_serious': 'severity adjustment', 'casualty_adjusted_severity_slight': 'severity adjustment',
    'collision_adjusted_severity_serious': 'severity adjustment', 'collision_adjusted_severity_slight': 'severity adjustment',
    'collision_injury_based': 'recording-system flag, reserved for subgroup checks',
    'did_police_officer_attend_scene_of_accident': 'possible proxy for injury severity (3.6.3)',
    'latitude': 'precise location (3.6.3)', 'longitude': 'precise location (3.6.3)',
    'location_easting_osgr': 'precise location (3.6.3)', 'location_northing_osgr': 'precise location (3.6.3)',
    'lsoa_of_accident_location': 'precise location (3.6.3)',
    'special_conditions_at_site': 'retired with the historic fields after 2023 (see availability table)',
}
HISTORIC = ['vehicle_manoeuvre_historic', 'vehicle_location_restricted_lane_historic', 'journey_purpose_of_driver_historic',
            'junction_detail_historic', 'pedestrian_crossing_human_control_historic',
            'pedestrian_crossing_physical_facilities_historic', 'carriageway_hazards_historic']
# Rule: a candidate field is dropped if it is missing or at least this share unknown in any one year.
UNAVAILABLE_SHARE = 0.90
FORBIDDEN_PATTERNS = ['severity', 'injury_based', 'police_officer', 'collision_index', 'latitude', 'longitude', 'lsoa', 'osgr']
print(len(PRIMARY), 'candidate predictors')

48 candidate predictors


In [3]:
# 3. Load eligible casualties and derive keys, outcome and time fields -------------
con = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT * FROM casualty_eligible', con)
con.close()
print('Eligible casualties:', len(df), df.groupby('source_year').size().to_dict())

df['cas_key'] = (df['collision_index'].astype(str) + '|' + df['vehicle_reference'].astype(str) + '|' + df['casualty_reference'].astype(str))
assert df['cas_key'].is_unique, 'cas_key is not unique'
df['group_id'] = df['source_year'].astype(str) + '_' + df['collision_index'].astype(str)
sev = pd.to_numeric(df['casualty_severity'])
assert sev.isin([1, 2, 3]).all()
df['ksi'] = (sev <= 2).astype(int)

dts = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
hhmm = df['time'].astype(str).str.extract(r'^(\d{1,2}):(\d{2})$').astype(float)
bad_dt = int(dts.isna().sum()); bad_t = int(hhmm[0].isna().sum())
print('Unparseable dates:', bad_dt, '| unparseable times:', bad_t)
assert bad_dt == 0 and bad_t == 0, 'Date or time format changed; stop and inspect.'
df['hour'] = hhmm[0].astype(int)
df['month'] = dts.dt.month.astype(int)
dow_from_date = ((dts.dt.dayofweek + 1) % 7 + 1)      # Sunday = 1 ... Saturday = 7, as in STATS19
agree = float((dow_from_date == pd.to_numeric(df['day_of_week'])).mean())
print(f'day_of_week field agrees with the date for {agree:.4%} of rows')
assert agree > 0.999, 'day_of_week does not match the date'
df['day_of_week'] = dow_from_date.astype(int)
assert df['hour'].between(0, 23).all() and df['month'].between(1, 12).all()
assert (dts.dt.year == df['source_year']).all(), 'date year differs from source_year'

Eligible casualties: 389132 {2023: 132977, 2024: 128272, 2025: 127883}
Unparseable dates: 0 | unparseable times: 0
day_of_week field agrees with the date for 100.0000% of rows


In [4]:
# 4. Availability check by year and the rule-based predictor list ------------------
def unknown_share(series, kind):
    s = pd.to_numeric(series, errors='coerce')
    return float(((s < 0) | s.isna()).mean())

rows = []
for name, tbl, kind in PRIMARY:
    for y in (DEV_YEARS + [TEST_YEAR]):
        sub = df[df['source_year'] == y]
        if name in sub.columns:
            share = unknown_share(sub[name], kind)
            n_levels = int(pd.to_numeric(sub[name], errors='coerce').nunique())
        else:
            share, n_levels = 1.0, 0
        rows.append(dict(field=name, table=tbl, kind=kind, year=y, unknown_share=round(share, 4), levels=n_levels))
avail = pd.DataFrame(rows)
for name in list(EXCLUDED_DESIGN) + HISTORIC:
    if name in df.columns:
        for y in (DEV_YEARS + [TEST_YEAR]):
            sub = df[df['source_year'] == y]
            avail.loc[len(avail)] = dict(field=name, table='excluded or retired', kind='-', year=y,
                                         unknown_share=round(unknown_share(sub[name], 'num'), 4),
                                         levels=int(pd.to_numeric(sub[name], errors='coerce').nunique()))
wide = avail.pivot_table(index=['field', 'table'], columns='year', values='unknown_share').reset_index()
wide.columns = ['field', 'table'] + [f'unknown_{c}' for c in wide.columns[2:]]
wide.to_csv(OUT / 'predictor_availability_by_year.csv', index=False)

prim = avail[avail['table'] != 'excluded or retired']
worst = prim.groupby('field')['unknown_share'].max()
rule_dropped = worst[worst >= UNAVAILABLE_SHARE].index.tolist()
print('Dropped by the availability rule:', rule_dropped or 'none')
print('Rule: field unavailable (at least 90% unknown) in any one year.')

# Predictor fields that are at least moderately unknown are listed for the dictionary note.
print('\nCandidate predictors with 20% or more unknown in any year:')
display(wide[wide['field'].isin(worst[worst >= 0.20].index)].round(3))
retired = wide[wide['field'].isin(HISTORIC)]
print('\nRetired *_historic fields (unknown share by year):')
display(retired.round(3))

Dropped by the availability rule: ['age_of_vehicle']
Rule: field unavailable (at least 90% unknown) in any one year.

Candidate predictors with 20% or more unknown in any year:


,field,table,unknown_2023,unknown_2024,unknown_2025
2,age_of_vehicle,vehicle,0.215,0.272,1.000
20,engine_capacity_cc,vehicle,0.230,0.291,0.249
30,junction_control,collision,0.435,0.435,0.453
49,propulsion_code,vehicle,0.215,0.272,0.220



Retired *_historic fields (unknown share by year):


,field,table,unknown_2023,unknown_2024,unknown_2025
6,carriageway_hazards_historic,excluded or retired,0.030,0.607,0.864
29,journey_purpose_of_driver_historic,excluded or retired,0.000,0.617,0.874
32,junction_detail_historic,excluded or retired,0.000,0.610,0.864
44,pedestrian_crossing_human_control_historic,excluded or retired,0.027,0.607,0.864
45,pedestrian_crossing_physical_facilities_historic,excluded or retired,0.027,0.607,0.864
66,vehicle_location_restricted_lane_historic,excluded or retired,0.010,0.607,0.864
68,vehicle_manoeuvre_historic,excluded or retired,0.010,0.622,0.877


In [5]:
# 5. Final predictor list and field dictionary -------------------------------------
final = [(n, t, k) for (n, t, k) in PRIMARY if n not in rule_dropped]
FEATURES = [n for n, _, _ in final]
KIND = {n: k for n, _, k in final}
NUM = [n for n in FEATURES if KIND[n] == 'num']
CAT = [n for n in FEATURES if KIND[n] == 'cat']

for n in FEATURES:                                         # leakage guard on names
    assert not any(p in n for p in FORBIDDEN_PATTERNS), f'forbidden predictor name: {n}'

dictionary = []
for n, t, k in PRIMARY:
    status = 'dropped by availability rule' if n in rule_dropped else ('primary (sensitivity: remove)' if n in SENSITIVITY_FIELDS else 'primary')
    dictionary.append(dict(field=n, table=t, kind=k, status=status,
                           note=SENSITIVITY_FIELDS.get(n, ''), importance_group=n))
for n, why in RESERVED.items():
    dictionary.append(dict(field=n, table='reserved', kind='-', status='reserved (not a primary predictor)', note=why, importance_group=''))
for n, why in EXCLUDED_DESIGN.items():
    dictionary.append(dict(field=n, table='excluded', kind='-', status='excluded', note=why, importance_group=''))
for n in HISTORIC:
    dictionary.append(dict(field=n, table='excluded', kind='-', status='excluded', note='historic field, mostly unknown after 2023; current field used instead', importance_group=''))
dict_df = pd.DataFrame(dictionary)
dict_df.to_csv(OUT / 'field_dictionary.csv', index=False)
print(f'{len(FEATURES)} primary predictors: {len(NUM)} numeric, {len(CAT)} categorical')
display(dict_df.groupby('status').size().rename('fields').to_frame())

# Level counts for every categorical predictor, to check against the data guide.
lev = []
for n in CAT:
    vc = pd.to_numeric(df[n], errors='coerce').value_counts(dropna=False).sort_index()
    for code_, cnt in vc.items():
        lev.append(dict(field=n, code=code_, casualties=int(cnt), share=round(cnt / len(df), 4)))
pd.DataFrame(lev).to_csv(OUT / 'categorical_level_counts.csv', index=False)

47 primary predictors: 8 numeric, 39 categorical


,fields
status,
dropped by availability rule,1
excluded,23
primary,46
primary (sensitivity: remove),1
reserved (not a primary predictor),3


In [6]:
# 6. Typing and deterministic recoding (nothing here learns from the data) ----------
X = pd.DataFrame(index=df.index)
for n in FEATURES:
    s = pd.to_numeric(df[n], errors='coerce')
    if KIND[n] == 'num':
        X[n] = s.where(s >= 0)                              # -1 (missing or out of range) -> NaN
        if n == 'age_of_casualty' or n == 'age_of_driver':
            X.loc[X[n] > 110, n] = np.nan                   # implausible ages -> missing
    else:
        X[n] = s.fillna(-1).astype('int32')                 # -1 stays a level
n_implausible = {n: int((pd.to_numeric(df[n], errors='coerce') > 110).sum()) for n in ('age_of_casualty', 'age_of_driver')}
print('Ages above 110 set to missing:', n_implausible)
print('\nNumeric missing share (all years):')
display(X[NUM].isna().mean().round(4).to_frame('missing_share'))

# Category levels present in 2025 but absent from 2023-2024 (for the drift log).
dev_mask = df['source_year'].isin(DEV_YEARS).values
new_levels = {}
for n in CAT:
    seen = set(X.loc[dev_mask, n].unique())
    new = sorted(set(X.loc[~dev_mask, n].unique()) - seen)
    if new:
        new_levels[n] = {int(v): int((X.loc[~dev_mask, n] == v).sum()) for v in new}
print('Categorical levels in 2025 that never occur in 2023-2024 (field: {code: casualties}):', new_levels or 'none')

Ages above 110 set to missing: {'age_of_casualty': 0, 'age_of_driver': 0}

Numeric missing share (all years):


,missing_share
age_of_casualty,0.0211
casualty_imd_decile,0.1206
age_of_driver,0.0570
engine_capacity_cc,0.2562
driver_imd_decile,0.1263
speed_limit,0.0000
number_of_vehicles,0.0000
number_of_casualties,0.0000


Categorical levels in 2025 that never occur in 2023-2024 (field: {code: casualties}): {'first_road_class': {-1: 8}, 'second_road_class': {9: 54}}


In [7]:
# 7. Development table, folds and pilot sample -------------------------------------
dev = df.loc[dev_mask, ['cas_key', 'group_id', 'collision_index', 'source_year', 'ksi']].reset_index(drop=True)
Xdev = X.loc[dev_mask].reset_index(drop=True)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
dev['fold'] = -1
for k, (_, test_idx) in enumerate(sgkf.split(Xdev, dev['ksi'], groups=dev['group_id'])):
    dev.loc[test_idx, 'fold'] = k
assert (dev['fold'] >= 0).all()

# Checks: each collision sits in exactly one fold; folds are balanced on size and KSI rate.
fold_per_group = dev.groupby('group_id')['fold'].nunique()
assert (fold_per_group == 1).all(), 'a collision appears in more than one fold'
fold_summary = (dev.groupby('fold').agg(casualties=('ksi', 'size'), collisions=('group_id', 'nunique'),
                                        ksi=('ksi', 'sum'), ksi_rate=('ksi', 'mean')).round(4))
fold_summary['share_2024'] = dev.assign(y24=(dev['source_year'] == 2024)).groupby('fold')['y24'].mean().round(3)
print('Folds (each collision in exactly one fold):')
display(fold_summary)
spread = fold_summary['ksi_rate'].max() - fold_summary['ksi_rate'].min()
print(f'KSI rate range across folds: {spread:.4f} | overall {dev["ksi"].mean():.4f}')

# Fixed 20% pilot of development collisions (all casualties of the chosen collisions).
rng = np.random.RandomState(SEED + 1)
groups = np.sort(dev['group_id'].unique())
pilot_groups = set(rng.choice(groups, size=int(round(PILOT_FRACTION * len(groups))), replace=False))
dev['pilot'] = dev['group_id'].isin(pilot_groups).astype(int)
pilot_summary = dev[dev['pilot'] == 1].agg(casualties=('ksi', 'size'))
print(f"\nPilot: {dev['pilot'].sum()} casualties in {len(pilot_groups)} collisions "
      f"({dev['pilot'].mean():.1%} of casualties), KSI rate {dev.loc[dev['pilot']==1, 'ksi'].mean():.4f}")
print('Pilot casualties per fold:', dev[dev['pilot'] == 1].groupby('fold').size().to_dict())

dev[['cas_key', 'collision_index', 'source_year', 'fold', 'pilot']].to_csv(DATA / 'folds_dev.csv', index=False)
fold_summary.to_csv(OUT / 'fold_summary.csv')

Folds (each collision in exactly one fold):


,casualties,collisions,ksi,ksi_rate,share_2024
fold,,,,,
0,52411,41006,11020,0.2103,0.491
1,52571,41194,11270,0.2144,0.490
2,51958,40924,11066,0.2130,0.492
3,52062,41049,10927,0.2099,0.491
4,52247,41012,11013,0.2108,0.491


KSI rate range across folds: 0.0045 | overall 0.2117

Pilot: 52254 casualties in 41037 collisions (20.0% of casualties), KSI rate 0.2133
Pilot casualties per fold: {0: 10659, 1: 10491, 2: 10329, 3: 10502, 4: 10273}


In [8]:
# 8. Save model inputs: development with labels, 2025 without labels ----------------
def save_frame(frame, name):
    try:
        frame.to_parquet(DATA / f'{name}.parquet', index=False); return f'{name}.parquet'
    except Exception as e:
        frame.to_pickle(DATA / f'{name}.pkl'); return f'{name}.pkl (parquet unavailable: {type(e).__name__})'

dev_out = pd.concat([dev[['cas_key', 'fold', 'pilot', 'ksi']], Xdev], axis=1)
test_X = X.loc[~dev_mask].reset_index(drop=True)
test_keys = df.loc[~dev_mask, ['cas_key']].reset_index(drop=True)
test_out = pd.concat([test_keys, test_X], axis=1)

# Leakage guards on what is written.
assert 'ksi' not in test_out.columns and 'fold' not in test_out.columns
for frame in (dev_out.drop(columns=['cas_key', 'fold', 'pilot', 'ksi']), test_out.drop(columns=['cas_key'])):
    assert list(frame.columns) == FEATURES
    assert not any(any(p in c for p in FORBIDDEN_PATTERNS) for c in frame.columns)

f1 = save_frame(dev_out, 'features_dev_2023_2024')
f2 = save_frame(test_out, 'features_test_2025_NO_LABELS')
print('Saved:', f1, '|', f2)
print('Development rows:', len(dev_out), '| 2025 rows (no labels):', len(test_out))

# Seal the 2025 labels in a separate folder with a checksum.
sealed = df.loc[~dev_mask, ['cas_key', 'ksi']].reset_index(drop=True)
sealed_path = SEALED / 'labels_test_2025_SEALED.csv'
if sealed_path.exists():
    print('Sealed file already exists; checksum compared, not overwritten.')
    assert hashlib.sha256(sealed_path.read_bytes()).hexdigest() == hashlib.sha256(sealed.to_csv(index=False).encode()).hexdigest(), \
        'Sealed labels differ from the saved file. Investigate before continuing.'
else:
    sealed.to_csv(sealed_path, index=False)
sha = hashlib.sha256(sealed_path.read_bytes()).hexdigest()
(SEALED / 'README.txt').write_text(
    'Do not open or load this folder until the freeze record is complete (methodology 3.7.2).\n'
    f'labels_test_2025_SEALED.csv sha256: {sha}\nWritten: {dt.datetime.now().isoformat(timespec="seconds")}\n')
print('Sealed 2025 labels sha256:', sha)

Saved: features_dev_2023_2024.parquet | features_test_2025_NO_LABELS.parquet
Development rows: 261249 | 2025 rows (no labels): 127883
Sealed 2025 labels sha256: 22ddae8fde110307daf9bd711a84f0b9cd18166d24e52194082b60e53cc3d4e3


In [9]:
# 9. Evaluation-only files: road-user group, recording system, force --------------------
MOTORCYCLE = {2, 3, 4, 5, 23, 97}          # motorcycle types incl. electric and unknown cc
def road_user(t):
    if t == 0: return 'pedestrian'
    if t == 1: return 'cyclist'
    if t in MOTORCYCLE: return 'motorcyclist'
    if t == 33: return 'e-scooter rider'      # casualty_type 33; matches escooter_flag in the data
    return 'other'
strata = df[['cas_key', 'source_year', 'police_force']].copy()
ct = pd.to_numeric(df['casualty_type'], errors='coerce')
strata['road_user_group'] = ct.map(road_user)
strata['injury_based'] = pd.to_numeric(df['casualty_injury_based'], errors='coerce')

fpath = LINK / 'forces_recording_system_2023_2025.csv'
if fpath.exists():
    forces = pd.read_csv(fpath)[['police_force', 'system_group']]
    strata = strata.merge(forces, on='police_force', how='left')
else:
    strata['system_group'] = np.nan
    print('WARNING: forces file from notebook 01 not found; system_group left empty.')

strata[strata['source_year'].isin(DEV_YEARS)].to_csv(DATA / 'eval_strata_dev.csv', index=False)
strata[strata['source_year'] == TEST_YEAR].to_csv(DATA / 'eval_strata_2025.csv', index=False)

print('Road-user groups: casualties by period; KSI rate for development only (2025 labels stay sealed):')
period = np.where(dev_mask, '2023-24', '2025')
tab = strata.assign(ksi=np.where(dev_mask, df['ksi'], np.nan), period=period).groupby(['period', 'road_user_group']).agg(
    casualties=('cas_key', 'size'), dev_ksi_rate=('ksi', 'mean')).round(3)
display(tab)
print('System group by period:'); display(strata.assign(period=np.where(dev_mask, '2023-24', '2025')).groupby(['period', 'system_group'], dropna=False).size().to_frame('casualties'))

Road-user groups: casualties by period; KSI rate for development only (2025 labels stay sealed):


casualties  dev_ksi_rate
period  road_user_group                          
2023-24 cyclist               29548         0.250
        e-scooter rider        2236         0.302
        motorcyclist          32938         0.332
        other                158088         0.153
        pedestrian            38439         0.313
2025    cyclist               16042           NaN
        e-scooter rider        1227           NaN
        motorcyclist          15447           NaN
        other                 75189           NaN
        pedestrian            19978           NaN

System group by period:


casualties
period  system_group                       
2023-24 changed or mixed              99927
        stable injury-based          144856
        stable non-injury-based       16466
2025    changed or mixed              49066
        stable injury-based           70735
        stable non-injury-based        8082

In [10]:
# 10. Summary record -------------------------------------------------------------------
summary = dict(
    created=dt.datetime.now().isoformat(timespec='seconds'), seed=SEED, folds=N_FOLDS, pilot_fraction=PILOT_FRACTION,
    sklearn=sklearn.__version__, pandas=pd.__version__,
    dev_casualties=int(len(dev_out)), dev_collisions=int(dev['group_id'].nunique()), dev_ksi_rate=round(float(dev['ksi'].mean()), 4),
    test_casualties_2025=int(len(test_out)),
    pilot_casualties=int(dev['pilot'].sum()), pilot_collisions=len(pilot_groups),
    predictors=len(FEATURES), numeric=NUM, categorical=CAT, dropped_by_rule=rule_dropped,
    sensitivity_field=list(SENSITIVITY_FIELDS), reserved_fields=list(RESERVED),
    new_levels_in_2025=new_levels, sealed_labels_sha256=sha,
)
(OUT / 'preprocessing_summary.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: v for k, v in summary.items() if k not in ('numeric', 'categorical')}, indent=2, default=str))
print('\nModel inputs:'); [print(' -', p.name) for p in sorted(DATA.glob('*'))]
print('\nNEXT: notebook 03 (pilot and tuning). It reads features_dev_2023_2024 and folds only. The sealed 2025 labels stay closed.')

{
  "created": "2026-11-07T12:04:29",
  "seed": 20261107,
  "folds": 5,
  "pilot_fraction": 0.2,
  "sklearn": "1.6.1",
  "pandas": "2.2.3",
  "dev_casualties": 261249,
  "dev_collisions": 205185,
  "dev_ksi_rate": 0.2117,
  "test_casualties_2025": 127883,
  "pilot_casualties": 52254,
  "pilot_collisions": 41037,
  "predictors": 47,
  "dropped_by_rule": [
    "age_of_vehicle"
  ],
  "sensitivity_field": [
    "number_of_casualties"
  ],
  "reserved_fields": [
    "police_force",
    "casualty_injury_based",
    "escooter_flag"
  ],
  "new_levels_in_2025": {
    "first_road_class": {
      "-1": 8
    },
    "second_road_class": {
      "9": 54
    }
  },
  "sealed_labels_sha256": "22ddae8fde110307daf9bd711a84f0b9cd18166d24e52194082b60e53cc3d4e3"
}

Model inputs:
 - eval_strata_2025.csv
 - eval_strata_dev.csv
 - features_dev_2023_2024.parquet
 - features_test_2025_NO_LABELS.parquet
 - folds_dev.csv

NEXT: notebook 03 (pilot and tuning). It reads features_dev_2023_2024 and folds only. The